# 동형암호(FHE) 원리 실습 — 【답안지】 순수 파이썬

**서울여대 GSPP Privacy Scholar Camp · 이승환 (waLLLnut / LatticA)**

`numpy` 없이 표준 라이브러리(`random`, `cmath`)만으로 돌립니다.
**토이는 $n=1$(비밀키 한 성분)** 이라 손으로 따라갈 수 있게 회로를 전부 펼쳐 적었고,
**★실습에서 $n=2$** 로 직접 3성분·9성분을 손으로 전개합니다.

> **주의:** 장난감 파라미터 — 실제 보안 파라미터가 아닙니다.


## 0. 파라미터
$q=10^6$, $\Delta=10^2$. 토이 비밀키는 **한 성분 $s$** (작게 잡아 손으로 따라가기).


In [ ]:
import random, cmath   # 파이썬 기본 내장

q = 10**6
Delta = 100
random.seed(42)

def center(x):
    "값을 (-q/2, q/2] 범위로 (부호 있는 대표원)"
    return ((int(x) + q // 2) % q) - q // 2

s = 3      # 비밀키 (단일 성분)
print("비밀키 s =", s)

## 1. LWE 암호화 / phase / 복호화 ($n=1$)
암호문 $=(c_0,c_1)=(b,\,-a)$. **phase** $=c_0\cdot1+c_1 s = b - a s = \Delta m + e$.


In [ ]:
def encrypt(m):
    a = random.randint(0, q - 1)
    e = random.randint(-3, 3)            # 곱셈까지 안정적으로 복호되게 작게
    b = (a * s + e + Delta * m) % q      # 회로: a*s + e + Δm
    return [b, (-a) % q]                  # (c0, c1)

def phase(c):
    return center(c[0] + c[1] * s)        # 회로: c0*1 + c1*s

def decrypt(c):
    return round(phase(c) / Delta)

c1 = encrypt(2)     # 메시지 2
c2 = encrypt(3)     # 메시지 3
print("암호문 c1 =", c1)
print("phase(c1) =", phase(c1), "-> 복호:", decrypt(c1))
print("phase(c2) =", phase(c2), "-> 복호:", decrypt(c2))
assert decrypt(c1) == 2 and decrypt(c2) == 3
print("OK: 암호화/복호화")

## 2. 덧셈 — 두 성분을 그대로 더한다

In [ ]:
# 회로: 성분별 덧셈 (두 개 직접)
c_add = [(c1[0] + c2[0]) % q,
         (c1[1] + c2[1]) % q]

print("phase(c1+c2) =", phase(c_add), "-> 복호:", decrypt(c_add), "(기대 5)")
assert decrypt(c_add) == 2 + 3
print("OK: 동형 덧셈")

### ★ 실습 1. 메시지와 오류를 바꾸면?
세 값만 바꿔 실행. 오류 한계가 $\Delta/2=50$ 에 가까워지면 언제 복호가 흔들릴까요?


In [ ]:
TRY_M1, TRY_M2 = 4, -1
TRY_ERROR_BOUND = 10

trng = random.Random(2026)

def encrypt_trial(m, bound):
    a = trng.randint(0, q - 1)
    e = trng.randint(-bound, bound)
    b = (a * s + e + Delta * m) % q
    return [b, (-a) % q], e

t1, e1 = encrypt_trial(TRY_M1, TRY_ERROR_BOUND)
t2, e2 = encrypt_trial(TRY_M2, TRY_ERROR_BOUND)
tadd = [(t1[0] + t2[0]) % q, (t1[1] + t2[1]) % q]

g1, g2, gadd = decrypt(t1), decrypt(t2), decrypt(tadd)
print("오류 (e1, e2):", (e1, e2), " / 합:", e1 + e2, " (기준 |합| <", Delta // 2, ")")
print("개별 복호:", (g1, g2), " / 예상:", (TRY_M1, TRY_M2))
print("덧셈 복호:", gadd, " / 예상:", TRY_M1 + TRY_M2)
if (g1, g2, gadd) == (TRY_M1, TRY_M2, TRY_M1 + TRY_M2):
    print("결과: 성공")
else:
    print("결과: 복호 실패 — 오류와 Delta의 상대적 크기를 확인하세요")

## 3. 곱셈 — 텐서곱 ($n=1$: 4성분 모두 직접)
확장키 $\bar{\mathbf s}=(1,s)$. 두 phase 의 곱이
$\langle\bar{\mathbf c}_1\otimes\bar{\mathbf c}_2,\ \bar{\mathbf s}\otimes\bar{\mathbf s}\rangle$.
키·암호문 텐서곱(각 $2\times2=4$ 성분)과 내적(4항)을 펼칩니다. 새 비밀 단항식은 $s^2$.


In [ ]:
sbar = [1, s]                       # 확장키 (1, s)
# 키 텐서곱  sbar ⊗ sbar = (1, s, s, s^2)
t = [1 * 1, 1 * s,
     s * 1, s * s]
# 암호문 텐서곱  c1 ⊗ c2 (mod q)
cm = [(c1[0] * c2[0]) % q, (c1[0] * c2[1]) % q,
      (c1[1] * c2[0]) % q, (c1[1] * c2[1]) % q]
# phase(곱) = 내적 (4항 직접)
ph_mul = center(cm[0]*t[0] + cm[1]*t[1] + cm[2]*t[2] + cm[3]*t[3])

print("텐서 키 (1, s, s, s^2) =", t)
print("phase(곱) =", ph_mul, " ≈ Delta^2*2*3 =", Delta**2 * 6)
print("Delta^2 로 나눠 복호:", round(ph_mul / Delta**2), "(기대 6)")
assert round(ph_mul / Delta**2) == 2 * 3
print("OK: 동형 곱셈 (2차 암호문, 새 비밀 s^2)")

### 선택 심화 3-1. 왜 하필 '텐서'인가
$\mu_1\mu_2=(c_0{+}c_1 s)(c_0'{+}c_1' s)
=c_0c_0'\cdot1+c_0c_1'\cdot s+c_1c_0'\cdot s+c_1c_1'\cdot s^2$
— 곱한 phase 가 곧 텐서 내적.


In [ ]:
mu1 = phase(c1)
mu2 = phase(c2)
print("(1) mu1 * mu2 =", center(mu1 * mu2), " / (2) 텐서 내적 =", ph_mul)
assert center(mu1 * mu2) == ph_mul
print("=> 곱한 phase = 텐서 내적. 텐서곱은 '키에 대한 이차형식'을 편 것.")

### ★ 실습 2. 이제 $n=2$ — 손으로 직접 전개
비밀키가 두 성분 $s_1,s_2$ 면 확장키 3성분, 텐서 9성분.
$n=1$ 을 흉내 내어 `None` 세 슬롯을 **손으로** 채우면 자동 검증됩니다.


In [ ]:
s1 = 2
s2 = 3
er = random.Random(7)

def encrypt2(m):
    a1 = er.randint(0, q - 1)
    a2 = er.randint(0, q - 1)
    e  = er.randint(-3, 3)
    b  = (a1 * s1 + a2 * s2 + e + Delta * m) % q
    return [b, (-a1) % q, (-a2) % q]

c1_2 = encrypt2(2)
c2_2 = encrypt2(3)

# ---- 손으로 채우기 (n=1 을 3성분·9성분으로) --------------------------------
sbar2 = [1, s1, s2]
t2    = [1*1, 1*s1, 1*s2,  s1*1, s1*s1, s1*s2,  s2*1, s2*s1, s2*s2]
cmul2 = [(c1_2[0]*c2_2[0])%q, (c1_2[0]*c2_2[1])%q, (c1_2[0]*c2_2[2])%q,
         (c1_2[1]*c2_2[0])%q, (c1_2[1]*c2_2[1])%q, (c1_2[1]*c2_2[2])%q,
         (c1_2[2]*c2_2[0])%q, (c1_2[2]*c2_2[1])%q, (c1_2[2]*c2_2[2])%q]
# --------------------------------------------------------------------------

if sbar2 is None or t2 is None or cmul2 is None:
    print("슬롯을 손으로 채우세요 (확장키 3성분, 텐서 9성분).")
    print("기대값 -> 확장키 3, 텐서 9, 덧셈 복호 5, 곱셈 복호 6")
else:
    ca = [(c1_2[0] + c2_2[0]) % q, (c1_2[1] + c2_2[1]) % q, (c1_2[2] + c2_2[2]) % q]
    ph_add = center(ca[0]*sbar2[0] + ca[1]*sbar2[1] + ca[2]*sbar2[2])
    ph_m2  = center(cmul2[0]*t2[0] + cmul2[1]*t2[1] + cmul2[2]*t2[2]
                  + cmul2[3]*t2[3] + cmul2[4]*t2[4] + cmul2[5]*t2[5]
                  + cmul2[6]*t2[6] + cmul2[7]*t2[7] + cmul2[8]*t2[8])
    print("확장키 차원:", len(sbar2), "(기대 3) / 텐서 차원:", len(t2), "(기대 9)")
    print("덧셈 복호:", round(ph_add / Delta), "(기대 5) / 곱셈 복호:", round(ph_m2 / Delta**2), "(기대 6)")
    assert len(sbar2) == 3 and len(t2) == 9
    assert round(ph_add / Delta) == 5 and round(ph_m2 / Delta**2) == 6
    print("OK: n=2 를 손으로 전개 완료 🎉")

## 4. 리니어라이제이션 --- **곱셈에서 나온 `cm` 을 그대로**
위 곱셈의 2차 암호문 $cm$(키 $1,s,s,s^2$)에서 새 비밀 $s^2$ 의 자리 계수는 $cm[3]$.
이 큰 수를 **가젯(10진 6자리)** 으로 쪼개 KSK 로 $(1,s)$ 암호문으로 치환합니다.
$s$-자리 둘($cm[1],cm[2]$)은 합쳐 하나로.


In [ ]:
def encrypt_raw(v):
    "스케일 없이 값 v 자체를 암호화 (phase = v). KSK 용."
    a = random.randint(0, q - 1)
    e = random.randint(-3, 3)
    return [(a * s + e + v) % q, (-a) % q]

# KSK: s^2 을 10진 각 자리(10^l)로 암호화 --- 자리마다 하나 (가젯이라 6개)
K0 = encrypt_raw(s*s * 1)
K1 = encrypt_raw(s*s * 10)
K2 = encrypt_raw(s*s * 100)
K3 = encrypt_raw(s*s * 1000)
K4 = encrypt_raw(s*s * 10000)
K5 = encrypt_raw(s*s * 100000)

# s^2 자리 계수 cm[3] 를 10진 6자리로 분해 (직접)
x  = cm[3] % q
g0 = x % 10
g1 = (x // 10) % 10
g2 = (x // 100) % 10
g3 = (x // 1000) % 10
g4 = (x // 10000) % 10
g5 = (x // 100000) % 10
print("cm[3] =", cm[3], " -> 10진 자릿수 [g0..g5] =", [g0, g1, g2, g3, g4, g5])

# 결합: 각 자리 digit * KSK (6항 직접) = Enc_s(cm[3]*s^2)
comb0 = (g0*K0[0] + g1*K1[0] + g2*K2[0] + g3*K3[0] + g4*K4[0] + g5*K5[0]) % q
comb1 = (g0*K0[1] + g1*K1[1] + g2*K2[1] + g3*K3[1] + g4*K4[1] + g5*K5[1]) % q

# s^2 항 제거 -> (1, s) 2성분:  c0 += comb0,  s-계수 = cm[1]+cm[2] + comb1
c_relin = [(cm[0] + comb0) % q, (cm[1] + cm[2] + comb1) % q]
ph_relin = phase(c_relin)
print("relin 후 암호문 =", c_relin, ", phase =", ph_relin)
print("복호:", round(ph_relin / Delta**2), "(기대 6, s^2 제거·phase 보존)")
assert round(ph_relin / Delta**2) == 6
print("OK: 리니어라이제이션 (곱셈 결과의 s^2 를 없애 다시 (1,s) 로)")

## 5. 리스케일 --- relin 결과 `c_relin` 을 $\div\Delta$
스케일이 $\Delta^2$ 이니 암호문을 $\Delta$ 로 나누고 **모듈러스도 $q\to q/\Delta$** 로 함께 줄입니다.


In [ ]:
q2 = q // Delta        # 새 모듈러스 10^4

# 두 성분을 Δ 로 나눠 반올림, 새 모듈러스로 (직접)
c_rs = [round(center(c_relin[0]) / Delta) % q2,
        round(center(c_relin[1]) / Delta) % q2]

# 새 모듈러스에서 phase = c0 + c1*s
ph_rs = ((c_rs[0] + c_rs[1] * s + q2 // 2) % q2) - q2 // 2
print("리스케일 후 암호문 =", c_rs, ", phase =", ph_rs, " (스케일 Δ)")
print("복호:", round(ph_rs / Delta), "(기대 6)")
assert round(ph_rs / Delta) == 6
print("OK: 리스케일 (Δ^2 -> Δ, 곱셈 결과 6)")

# 2부. NTT — 다항식 곱셈을 빠르게
링 $\mathbb Z_{17}[X]/(X^4-1)$ 에서 순환 합성곱을 **스쿨북 = FFT = NTT** 로.
$N=4$ 라 4점 변환을 전부 펼칩니다.


In [ ]:
P = 17
a = [1, 2, 3, 4]
b = [5, 6, 7, 8]
print("4^1..4^4 mod 17 =", [pow(4, 1, P), pow(4, 2, P), pow(4, 3, P), pow(4, 4, P)], "(원시 4차 근)")

## 6. 스쿨북 순환 합성곱 ($X^4\equiv1$ 이라 인덱스 $\bmod 4$, 네 출력 직접)

In [ ]:
# res[k] = 합 (i+j ≡ k mod 4) a[i]*b[j]  — 네 개를 그대로
s0 = (a[0]*b[0] + a[1]*b[3] + a[2]*b[2] + a[3]*b[1]) % P
s1_ = (a[0]*b[1] + a[1]*b[0] + a[2]*b[3] + a[3]*b[2]) % P
s2_ = (a[0]*b[2] + a[1]*b[1] + a[2]*b[0] + a[3]*b[3]) % P
s3_ = (a[0]*b[3] + a[1]*b[2] + a[2]*b[1] + a[3]*b[0]) % P
c_school = [s0, s1_, s2_, s3_]
print("스쿨북 순환 합성곱 mod 17 =", c_school)

## 7. FFT --- 복소수로 돌고, 마지막에 mod 17
4점 DFT(트위들 $1,-i,-1,i$). 주파수영역은 **복소수**(실수부/허수부)라 근사 →
IFFT 뒤 실수부를 반올림해 정수 순환합성곱을 얻고 **mod 17**. 값들을 다 찍어봅니다.


In [ ]:
def dft4(x):
    X0 = x[0] + x[1] + x[2] + x[3]
    X1 = x[0] - 1j*x[1] - x[2] + 1j*x[3]
    X2 = x[0] - x[1] + x[2] - x[3]
    X3 = x[0] + 1j*x[1] - x[2] - 1j*x[3]
    return [X0, X1, X2, X3]

def idft4(X):
    x0 = (X[0] + X[1] + X[2] + X[3]) / 4
    x1 = (X[0] + 1j*X[1] - X[2] - 1j*X[3]) / 4
    x2 = (X[0] - X[1] + X[2] - X[3]) / 4
    x3 = (X[0] - 1j*X[1] - X[2] + 1j*X[3]) / 4
    return [x0, x1, x2, x3]

def sc(z):
    "복소수를  실수부 +허수부 i  형태로"
    return "%+.2f%+.2fi" % (z.real, z.imag)

Fa = dft4(a)
Fb = dft4(b)
print("복소 DFT(a):", sc(Fa[0]), sc(Fa[1]), sc(Fa[2]), sc(Fa[3]))
print("복소 DFT(b):", sc(Fb[0]), sc(Fb[1]), sc(Fb[2]), sc(Fb[3]))

Fp = [Fa[0]*Fb[0], Fa[1]*Fb[1], Fa[2]*Fb[2], Fa[3]*Fb[3]]
print("포인트와이즈 곱(복소):", sc(Fp[0]), sc(Fp[1]), sc(Fp[2]), sc(Fp[3]))

inv = idft4(Fp)
print("IFFT(복소, 허수부≈0):", sc(inv[0]), sc(inv[1]), sc(inv[2]), sc(inv[3]))

conv = [round(inv[0].real), round(inv[1].real), round(inv[2].real), round(inv[3].real)]
print("실수부 반올림(정수 순환합성곱):", conv)

c_fft = [conv[0] % P, conv[1] % P, conv[2] % P, conv[3] % P]
print("mod 17 ->", c_fft)
assert c_fft == c_school
print("OK: 스쿨북 == FFT (복소로 돌고 마지막에 mod p)")

## 8. NTT --- **mod 17 위의 FFT** (주파수영역도 정수)
원시근 $w=4$ 로 만든 4점 변환. 복소수 대신 **처음부터 끝까지 정수 mod 17** →
주파수영역 값도 정수라 반올림이 필요 없습니다. $W_{ij}=4^{ij}\bmod17$, $w^{-1}=13$, $N^{-1}=13$.


In [ ]:
def ntt(x):
    A0 = (x[0] +      x[1] +      x[2] +      x[3]) % P
    A1 = (x[0] +  4 * x[1] + 16 * x[2] + 13 * x[3]) % P
    A2 = (x[0] + 16 * x[1] +      x[2] + 16 * x[3]) % P
    A3 = (x[0] + 13 * x[1] + 16 * x[2] +  4 * x[3]) % P
    return [A0, A1, A2, A3]

def intt(X):
    a0 = (13 * (X[0] +      X[1] +      X[2] +      X[3])) % P
    a1 = (13 * (X[0] + 13 * X[1] + 16 * X[2] +  4 * X[3])) % P
    a2 = (13 * (X[0] + 16 * X[1] +      X[2] + 16 * X[3])) % P
    a3 = (13 * (X[0] +  4 * X[1] + 16 * X[2] + 13 * X[3])) % P
    return [a0, a1, a2, a3]

Na = ntt(a)
Nb = ntt(b)
print("NTT(a) =", Na, "  NTT(b) =", Nb, "  (주파수영역 = 정수 mod 17)")

# 주파수영역에서 포인트와이즈 곱을 mod 17
Np = [(Na[0]*Nb[0]) % P, (Na[1]*Nb[1]) % P, (Na[2]*Nb[2]) % P, (Na[3]*Nb[3]) % P]
print("주파수영역 곱 mod 17 :", Np)

c_ntt = intt(Np)
print("INTT ->", c_ntt)
assert c_ntt == c_school
print("OK: NTT == 스쿨북 (mod 17 위의 FFT, 반올림 없이 정확)")

## 9. 종합 --- schoolbook = FFT = NTT ( = mod p 위의 FFT )

In [ ]:
print("스쿨북            :", c_school)
print("FFT (복소→mod17)  :", c_fft)
print("NTT (정수 mod17)  :", c_ntt)
assert c_school == c_fft == c_ntt
print()
print("=> 셋 다 동일.")
print("   FFT: 복소수로 돌아 근사 -> 실수부 반올림 -> mod p (마지막에)")
print("   NTT: 주파수영역도 정수 mod p -> 반올림 없이 정확 ( = mod p 위의 FFT )")

## 10. 주파수 영역에서 mod 17 하고 역변환해도 그대로
NTT 는 **주파수영역 값을 mod 17 로 다뤄도**(포인트와이즈 곱을 mod 17) 역변환 뒤
원래 값이 mod 17 로 정확히 복원됩니다 — 그래서 FFT 와 달리 오차가 쌓이지 않습니다.


In [ ]:
A = ntt(a)
A_modp = [A[0] % P, A[1] % P, A[2] % P, A[3] % P]   # 주파수영역에서 mod 17
back = intt(A_modp)                                  # 역변환
back_modp = [back[0] % P, back[1] % P, back[2] % P, back[3] % P]   # 다시 mod 17
a_mod = [a[0] % P, a[1] % P, a[2] % P, a[3] % P]
print("NTT(a)              =", A)
print("주파수영역 mod 17   =", A_modp)
print("INTT 후 mod 17      =", back_modp, " (원래 a mod 17 =", a_mod, ")")
assert back_modp == a_mod
print("OK: 주파수영역 mod p -> 역변환 -> mod p 해도 값 그대로")

### ★ 실습 3. 단위근을 바꾸면?
`TRY_W=4` 는 원시 4차 근(차수 4). `TRY_W=2` 면 차수 8 이라 왕복이 깨집니다.


In [ ]:
TRY_W = 4

# TRY_W 의 거듭제곱 4개 직접: 원시 4차 근이면 ^4 에서 처음 1
p1 = pow(TRY_W, 1, P)
p2 = pow(TRY_W, 2, P)
p3 = pow(TRY_W, 3, P)
p4 = pow(TRY_W, 4, P)
is_primitive4 = (p4 == 1 and p2 != 1 and p1 != 1)

winv = pow(TRY_W, -1, P)
Ninv = pow(4, -1, P)

def W(i, j):  return pow(TRY_W, i * j, P)
A0 = (W(0,0)*a[0] + W(0,1)*a[1] + W(0,2)*a[2] + W(0,3)*a[3]) % P
A1 = (W(1,0)*a[0] + W(1,1)*a[1] + W(1,2)*a[2] + W(1,3)*a[3]) % P
A2 = (W(2,0)*a[0] + W(2,1)*a[1] + W(2,2)*a[2] + W(2,3)*a[3]) % P
A3 = (W(3,0)*a[0] + W(3,1)*a[1] + W(3,2)*a[2] + W(3,3)*a[3]) % P

def Wi(i, j): return pow(winv, i * j, P)
b0 = (Ninv*(Wi(0,0)*A0 + Wi(0,1)*A1 + Wi(0,2)*A2 + Wi(0,3)*A3)) % P
b1 = (Ninv*(Wi(1,0)*A0 + Wi(1,1)*A1 + Wi(1,2)*A2 + Wi(1,3)*A3)) % P
b2 = (Ninv*(Wi(2,0)*A0 + Wi(2,1)*A1 + Wi(2,2)*A2 + Wi(2,3)*A3)) % P
b3 = (Ninv*(Wi(3,0)*A0 + Wi(3,1)*A1 + Wi(3,2)*A2 + Wi(3,3)*A3)) % P
back = [b0, b1, b2, b3]
a_mod = [a[0] % P, a[1] % P, a[2] % P, a[3] % P]

print("TRY_W 거듭제곱 [^1,^2,^3,^4]:", [p1, p2, p3, p4])
print("원시 4차 근인가? (^4 에서 처음 1)", is_primitive4)
print("왕복 결과:", back, " / 원래:", a_mod)
print("왕복 성공?", back == a_mod)

### 정리
- 순환 합성곱은 **스쿨북 = FFT = NTT** 로 모두 같은 결과.
- 토이($n=1$)는 회로를 전부 펼쳐 어떤 곱·합이 일어나는지 직접 보였고, ★실습에서 $n=2$ 로 손수 확장했습니다.
- $n,N$ 이 커지면(실전) 이 펼친 식이 **for-루프**가 됩니다.

**수고하셨습니다! 🎉**


## 부록. 답안 (강사용)

**★ 실습 2 — n=2** (손으로 채운 슬롯)
```python
sbar2 = [1, s1, s2]
t2    = [1*1, 1*s1, 1*s2,  s1*1, s1*s1, s1*s2,  s2*1, s2*s1, s2*s2]
cmul2 = [(c1_2[0]*c2_2[0])%q, (c1_2[0]*c2_2[1])%q, (c1_2[0]*c2_2[2])%q,
         (c1_2[1]*c2_2[0])%q, (c1_2[1]*c2_2[1])%q, (c1_2[1]*c2_2[2])%q,
         (c1_2[2]*c2_2[0])%q, (c1_2[2]*c2_2[1])%q, (c1_2[2]*c2_2[2])%q]
```
출력: 확장키 3, 텐서 9, 덧셈 복호 5, 곱셈 복호 6.

**★ 실습 1** 오류 한계를 키워 $|e_1+e_2|\ge\Delta/2=50$ 이면 덧셈 복호부터 실패.
**★ 실습 3** `TRY_W=4` 성공(차수 4). `TRY_W=2` 는 차수 8(≠4)이라 원시 4차 근이 아니어서 왕복 실패.
